<font size = 4>Movies_Recommendation_System using Collaborative filtering</font>


 

In [216]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [217]:
#cosine_similarity(A, B) gives how similar two vectors are based on the angle between them, not their magnitude.
cosine_similarity(np.array( [[1,2,3,4]] ),np.array( [[2,3,4,5]] ) )   
# +1 → exactly same direction  
#  0 → orthogonal (no similarity)
# -1 → opposite direction

array([[0.99380799]])

In [218]:
cosine_similarity(np.array( [[1,1]] ),np.array( [[-1,-1]] ) )

array([[-1.]])

In [219]:
arr  =np.array( [[4,0,0],
       [2,3,0],
       [3.5,4,3]])
arr

array([[4. , 0. , 0. ],
       [2. , 3. , 0. ],
       [3.5, 4. , 3. ]])

In [220]:
cosine_similarity(arr, arr)    #array must be 2d or nd

array([[1.        , 0.5547002 , 0.57346234],
       [0.5547002 , 1.        , 0.8634134 ],
       [0.57346234, 0.8634134 , 1.        ]])

In [221]:
A = np.array([1, 2, 3, 4, 5])
B = np.array([2, 3, 4, 5, 6])

# Cosine similarity
cos_sim = cosine_similarity([A],[B])  # [ ] because cosine_similarity takes 2d array
print("Cosine Similarity:", cos_sim[0][0])

Cosine Similarity: 0.994936676326182


In [222]:
import pandas as pd 
import seaborn as sns
import datetime as dt
from datetime import datetime

In [223]:
data=pd.read_csv('rating.csv')
data.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [224]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [225]:
data.shape

(100836, 4)

In [226]:
data.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [227]:
data.duplicated().sum()

0

In [228]:
data['date']=pd.to_datetime(data['timestamp'],unit='s')
data.head()

,userId,movieId,rating,timestamp,date
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51


In [229]:
data.nunique()

userId         610
movieId       9724
rating          10
timestamp    85043
date         85043
dtype: int64

In [230]:
# no of reviews given by each user
data.userId.value_counts()  

userId
414    2698
599    2478
474    2108
448    1864
274    1346
       ... 
442      20
569      20
320      20
576      20
53       20
Name: count, Length: 610, dtype: int64

In [231]:
# check data integrity
data[['userId','movieId']].duplicated().sum()

0

In [232]:
# no of times a movie rated
data.movieId.value_counts()       

movieId
356       329
318       317
296       307
593       279
2571      278
         ... 
86279       1
86922       1
5962        1
87660       1
163981      1
Name: count, Length: 9724, dtype: int64

In [233]:
# number of ratings given by each user
data.groupby('userId').movieId.count()

userId
1       232
2        29
3        39
4       216
5        44
       ... 
606    1115
607     187
608     831
609      37
610    1302
Name: movieId, Length: 610, dtype: int64

In [234]:
print(sorted(data['date'].dt.year.unique()))

[1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]


In [235]:
new_df=data[data['date'].dt.year>=2010].sort_values(by=['timestamp']) 
new_df.reset_index(drop=True,inplace=True)
new_df

,userId,movieId,rating,timestamp,date
0,448,58803,4.0,1262337884,2010-01-01 09:24:44
1,292,68358,4.0,1262376359,2010-01-01 20:05:59
2,292,6377,3.5,1262376394,2010-01-01 20:06:34
3,292,69122,3.5,1262376411,2010-01-01 20:06:51
4,292,4306,4.0,1262376466,2010-01-01 20:07:46
...,...,...,...,...,...
39680,514,187031,2.5,1537674927,2018-09-23 03:55:27
39681,514,187595,3.0,1537674946,2018-09-23 03:55:46
39682,514,5247,2.5,1537757040,2018-09-24 02:44:00
39683,514,5246,1.5,1537757059,2018-09-24 02:44:19


In [236]:
new_df.nunique()

userId         251
movieId       7063
rating          10
timestamp    39187
date         39187
dtype: int64

In [237]:
rating_df=pd.pivot_table(data=new_df,index='userId',columns='movieId',values='rating', fill_value= 0)
rating_df

movieId,1,2,3,5,6,7,9,10,11,12,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,3.0,2.5,1.5,0.0,4.5,2.5,1.5,3.5,2.5,1.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
601,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
605,4.0,3.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [238]:
#print(sorted(rating_df[rating_df.index==2].values[0],reverse=True))

In [239]:
user_sim_df=cosine_similarity(rating_df,rating_df)
user_sim_df=pd.DataFrame(user_sim_df)
user_sim_df

,0,1,2,3,4,5,6,7,8,9,...,241,242,243,244,245,246,247,248,249,250
0,1.000000,0.000000,0.067445,0.119778,0.093728,0.103755,0.166253,0.090880,0.144635,0.129902,...,0.127551,0.089562,0.000000,0.104568,0.048508,0.098000,0.202671,0.000000,0.000000,0.102427
1,0.000000,1.000000,0.000000,0.017251,0.032299,0.009813,0.028241,0.004017,0.003070,0.002592,...,0.000000,0.000986,0.004230,0.025873,0.000000,0.039539,0.005048,0.010694,0.000000,0.032119
2,0.067445,0.000000,1.000000,0.108254,0.098190,0.072358,0.115139,0.210293,0.132590,0.156054,...,0.021316,0.166634,0.071183,0.149926,0.028088,0.090037,0.187179,0.104130,0.010328,0.121072
3,0.119778,0.017251,0.108254,1.000000,0.231673,0.305070,0.306971,0.288000,0.181460,0.227126,...,0.098416,0.279029,0.032932,0.308689,0.030566,0.206120,0.327609,0.131857,0.062907,0.249504
4,0.093728,0.032299,0.098190,0.231673,1.000000,0.456096,0.303651,0.114633,0.162771,0.172490,...,0.173957,0.124794,0.042647,0.202409,0.068444,0.210346,0.336671,0.082135,0.000000,0.196072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,0.098000,0.039539,0.090037,0.206120,0.210346,0.243616,0.344374,0.257201,0.174688,0.198093,...,0.163268,0.191467,0.129856,0.281788,0.044614,1.000000,0.179483,0.158966,0.029572,0.395932
247,0.202671,0.005048,0.187179,0.327609,0.336671,0.332570,0.319642,0.229425,0.255793,0.192193,...,0.127821,0.267318,0.072841,0.329780,0.141751,0.179483,1.000000,0.118538,0.070418,0.222491
248,0.000000,0.010694,0.104130,0.131857,0.082135,0.174219,0.161859,0.166505,0.070057,0.053658,...,0.012184,0.165494,0.063759,0.186431,0.054468,0.158966,0.118538,1.000000,0.054139,0.119295
249,0.000000,0.000000,0.010328,0.062907,0.000000,0.048965,0.059867,0.081971,0.018226,0.000000,...,0.033623,0.112182,0.015457,0.048115,0.000000,0.029572,0.070418,0.054139,1.000000,0.046809


In [240]:
user_sim_df.shape

(251, 251)

In [241]:
# due to above code indexes are reseted because of converiting array to dataframe so to take out the same index  

In [242]:
user_id_mapping=dict(zip(user_sim_df.index.values,rating_df.index.values))
user_id_mapping

{0: 2,
 1: 3,
 2: 10,
 3: 15,
 4: 16,
 5: 17,
 6: 18,
 7: 21,
 8: 22,
 9: 24,
 10: 25,
 11: 28,
 12: 29,
 13: 30,
 14: 41,
 15: 47,
 16: 49,
 17: 50,
 18: 52,
 19: 55,
 20: 60,
 21: 62,
 22: 63,
 23: 65,
 24: 67,
 25: 68,
 26: 70,
 27: 73,
 28: 76,
 29: 77,
 30: 80,
 31: 83,
 32: 86,
 33: 87,
 34: 88,
 35: 89,
 36: 92,
 37: 98,
 38: 103,
 39: 104,
 40: 105,
 41: 106,
 42: 111,
 43: 112,
 44: 114,
 45: 116,
 46: 119,
 47: 122,
 48: 123,
 49: 124,
 50: 125,
 51: 127,
 52: 131,
 53: 132,
 54: 139,
 55: 141,
 56: 143,
 57: 146,
 58: 148,
 59: 152,
 60: 153,
 61: 154,
 62: 158,
 63: 159,
 64: 168,
 65: 172,
 66: 177,
 67: 180,
 68: 184,
 69: 189,
 70: 190,
 71: 193,
 72: 196,
 73: 199,
 74: 203,
 75: 204,
 76: 205,
 77: 209,
 78: 210,
 79: 211,
 80: 212,
 81: 213,
 82: 220,
 83: 222,
 84: 227,
 85: 228,
 86: 231,
 87: 232,
 88: 233,
 89: 237,
 90: 241,
 91: 246,
 92: 247,
 93: 248,
 94: 249,
 95: 251,
 96: 252,
 97: 253,
 98: 256,
 99: 258,
 100: 261,
 101: 272,
 102: 274,
 103: 279,
 104: 

In [203]:
user_sim_df.index = rating_df.index
user_sim_df.columns = rating_df.index

In [204]:
user_sim_df

userId,2,3,10,15,16,17,18,21,22,24,...,585,586,590,596,598,599,601,605,606,610
userId,,,,,,,,,,,,,,,,,,,,,
2,1.000000,0.000000,0.067445,0.119778,0.093728,0.103755,0.166253,0.090880,0.144635,0.129902,...,0.127551,0.089562,0.000000,0.104568,0.048508,0.098000,0.202671,0.000000,0.000000,0.102427
3,0.000000,1.000000,0.000000,0.017251,0.032299,0.009813,0.028241,0.004017,0.003070,0.002592,...,0.000000,0.000986,0.004230,0.025873,0.000000,0.039539,0.005048,0.010694,0.000000,0.032119
10,0.067445,0.000000,1.000000,0.108254,0.098190,0.072358,0.115139,0.210293,0.132590,0.156054,...,0.021316,0.166634,0.071183,0.149926,0.028088,0.090037,0.187179,0.104130,0.010328,0.121072
15,0.119778,0.017251,0.108254,1.000000,0.231673,0.305070,0.306971,0.288000,0.181460,0.227126,...,0.098416,0.279029,0.032932,0.308689,0.030566,0.206120,0.327609,0.131857,0.062907,0.249504
16,0.093728,0.032299,0.098190,0.231673,1.000000,0.456096,0.303651,0.114633,0.162771,0.172490,...,0.173957,0.124794,0.042647,0.202409,0.068444,0.210346,0.336671,0.082135,0.000000,0.196072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,0.098000,0.039539,0.090037,0.206120,0.210346,0.243616,0.344374,0.257201,0.174688,0.198093,...,0.163268,0.191467,0.129856,0.281788,0.044614,1.000000,0.179483,0.158966,0.029572,0.395932
601,0.202671,0.005048,0.187179,0.327609,0.336671,0.332570,0.319642,0.229425,0.255793,0.192193,...,0.127821,0.267318,0.072841,0.329780,0.141751,0.179483,1.000000,0.118538,0.070418,0.222491
605,0.000000,0.010694,0.104130,0.131857,0.082135,0.174219,0.161859,0.166505,0.070057,0.053658,...,0.012184,0.165494,0.063759,0.186431,0.054468,0.158966,0.118538,1.000000,0.054139,0.119295


In [205]:
# give similarity of each user with respect of other users 
user2=user_sim_df[user_sim_df.index==2]
user2

userId,2,3,10,15,16,17,18,21,22,24,...,585,586,590,596,598,599,601,605,606,610
userId,,,,,,,,,,,,,,,,,,,,,
2,1.0,0.0,0.067445,0.119778,0.093728,0.103755,0.166253,0.09088,0.144635,0.129902,...,0.127551,0.089562,0.0,0.104568,0.048508,0.098,0.202671,0.0,0.0,0.102427


In [206]:
# sort the index in the ascending order of values, first index is values is smallest and last is largest
np.argsort(user2.values)

array([[200,  67,  57, 204,  73,  51,  82,  39,  97,  36,  33, 227, 176,
        169, 236, 237, 107, 249,   1, 150, 248, 148, 145, 143, 138, 243,
         11, 156,  62, 133,  35, 168,  56, 232,  78, 124, 154,  93,  71,
        132, 115,  49, 202,  64,  96, 209,  81,  41, 217, 165,  85, 228,
        196,  99, 180, 220, 106, 174, 121, 222, 108, 240, 211, 152,  60,
        184, 144,  26,  58, 215,  45, 104, 155, 245, 198, 221, 157, 194,
         19, 131,  91,  65, 109, 116, 147, 203,  74, 218, 177,   2,  15,
        205,  95, 127, 129, 120, 187,  87, 210, 233, 123, 101,  37, 189,
        201, 140, 163, 214, 231,  44,  75, 128,  70,  63,  20, 242, 167,
        102,   7, 234,  32, 226, 137, 192, 118,   4, 134, 191,  86, 195,
        246,  89,  66,  61, 170,  31, 250,  16,   5,  12, 244, 185,  90,
        164,  54,  72, 149, 186,  25, 207, 193, 130,  98, 125, 151, 190,
        139,  17, 161,  24,  52, 111,  76,  68, 179, 178,   3, 223, 172,
         50, 159, 182,  53, 241,  14, 160, 100,   9

In [207]:
# we did not consider the last values in our consideration to find similarity because last is the same user corresponding to which we are finding similar users
list(np.argsort(user2.values)[0])[-4:-1][::-1] 

[142, 162, 146]

In [208]:
# similar users to user 2 are:
user_id_mapping[142],user_id_mapping[162],user_id_mapping[146]           # use this because argsort give indexes start from 0 to n

(366, 417, 378)

In [209]:
# movies watched by user 2 
new_df[new_df.userId==2].movieId.values

array([   318,  79132, 131724, 115713,  99114, 112552,   3578,  91529,
        74458,  91658,  77455,   6874,   8798, 106782,  71535,  60756,
        46970,    333,  48516,  58559, 109487,  68157,  86345,  80906,
        89774,   1704, 122882, 114060,  80489], dtype=int64)

In [210]:
# similar user of user 2 is user 366 and movies watched by user 366
new_df[new_df.userId==366].movieId.values

array([ 58559,   7153,   4993,   6942,   6537,   5481, 109487, 116797,
       115617,    589,  79132,   1036,   2959,  33794,   2028,    110,
         6874,   7438,   1089,  48516,  91529,   3949,   8961,   6539,
         3578,  32587,  44191,   8368,  59315, 122882,  68157],
      dtype=int64)

In [211]:
sm_usr2=set(new_df[new_df.userId==2].movieId.values)

In [212]:
sm_usr366=set(new_df[new_df.userId==366].movieId.values)

In [213]:
# movies that both users have watched 
sm_usr2.intersection(sm_usr366)    

{3578, 6874, 48516, 58559, 68157, 79132, 91529, 109487, 122882}

In [214]:
# movies that user 366 has not watched and to recommond to him   
sm_usr366-sm_usr2        

{110,
 589,
 1036,
 1089,
 2028,
 2959,
 3949,
 4993,
 5481,
 6537,
 6539,
 6942,
 7153,
 7438,
 8368,
 8961,
 32587,
 33794,
 44191,
 59315,
 115617,
 116797}

In [285]:
# approach 1
for i in [2, 3, 16]:
    print('Recommended movies for user', i)

    # remove self similarity
    user_sim_score = user_sim_df.loc[i].drop(i)

    # get top 2 similar users (actual user IDs)
    top_users = user_sim_score.sort_values(ascending=False).head(3).index
    print("Top similar users:", list(top_users))

    for usr2 in top_users:
        print('Movie to recommend to user', i, 'based on user', usr2)

        sm_1st = set(new_df[new_df['userId'] == i].movieId.values)
        sm_2nd = set(new_df[new_df['userId'] == usr2].movieId.values)

        movies_to_recommend = list(sm_2nd - sm_1st)

        print(movies_to_recommend[:3])

    print('-----------------------------------------')

Recommended movies for user 2
Top similar users: [63, 56, 232]
Movie to recommend to user 2 based on user 63
[54272, 1, 33794]
Movie to recommend to user 2 based on user 56
[]
Movie to recommend to user 2 based on user 232
[62081, 78209, 58627]
-----------------------------------------
Recommended movies for user 3
Top similar users: [92, 103, 172]
Movie to recommend to user 3 based on user 92
[1801, 2572, 2454]
Movie to recommend to user 3 based on user 103
[1, 2, 122882]
Movie to recommend to user 3 based on user 172
[1210, 2762, 1580]
-----------------------------------------
Recommended movies for user 16
Top similar users: [174, 119, 111]
Movie to recommend to user 16 based on user 174
[]
Movie to recommend to user 16 based on user 119
[54272, 1, 69122]
Movie to recommend to user 16 based on user 111
[122882, 157699, 5]
-----------------------------------------
